In [ ]:
import os
import cv2
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedKFold

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print(f"✅ GPU ready: {gpus[0].name} | mixed_float16 enabled")
else:
    print("⚠️  No GPU found — running on CPU")

# ── Paths ────────────────────────────────────────────────────────────────────
CACHE_DIR  = r'C:\Users\Shafique\Documents\uni\ml project\cache'
IMAGES_DIR = r'C:\Users\Shafique\Documents\uni\ml project\dataset\final dataset\Images'
VAL_DIR    = os.path.join(IMAGES_DIR, 'val')

BATCH_SIZE = 32
EPOCHS     = 40
IMG_H, IMG_W = 224, 224

MODELS = [
    (r'C:\Users\Shafique\Documents\uni\ml project\cnn4\final_model.keras',
     'CNN (K-Means seg)',  'ch-saturation_k3_224x224'),
    (r"C:\Users\Shafique\Documents\uni\ml project\cnn5\final_model.keras",
     'CNN (Contour)', 'contour_224x224'),
     (r"C:\Users\Shafique\Documents\uni\ml project\cnn no seg\final_model.keras",
      'CNN (No Seg)', 'noseg_224x224'),
      (r"C:\Users\Shafique\Documents\uni\ml project\inception\final_model.keras",
       'Inception (K-Means)', 'inception_ch-saturation_k3_224x224'),
       (r"C:\Users\Shafique\Documents\uni\ml project\inception contour\final_model.keras",
        'Inception (Contour)', 'inception_contour_224x224'),
        (r"C:\Users\Shafique\Documents\uni\ml project\inception no seg\final_model.keras",
         'Inception (No Seg)', 'inception_noseg_224x224'),
         (r"C:\Users\Shafique\Documents\uni\ml project\densenet\final_model.keras",
          'DenseNet (K-Means)', 'densenet_ch-saturation_k3_224x224'),
          (r"C:\Users\Shafique\Documents\uni\ml project\densenet contour\final_model.keras",
           'DenseNet (K-Means)', "densenet_contour_224x224"),
           (r"C:\Users\Shafique\Documents\uni\ml project\densenet no seg\final_model.keras",
            "DenseNet (No Seg)", "densenet_noseg_224x224")
]

In [ ]:
def load_npz(path):
    d = np.load(path)
    return d['data'] if 'data' in d else d[list(d.keys())[0]]

def load_val_images(val_dir, n=50):
    """First n sorted images from val/Healthy and val/Diseased, normalised to [0,1]."""
    X, y = [], []
    for label, cls in [(0, 'Healthy'), (1, 'Diseased')]:
        cls_dir = os.path.join(val_dir, cls)
        files = sorted([
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])[:n]
        for fname in files:
            fpath = os.path.join(cls_dir, fname)
            img = cv2.imread(fpath)
            if img is None:
                print(f"  ⚠️  Could not read {fpath} — skipping")
                continue
            img = cv2.cvtColor(cv2.resize(img, (IMG_W, IMG_H)), cv2.COLOR_BGR2RGB)
            X.append(img.astype(np.float32) / 255.0)
            y.append(label)
    return np.array(X), np.array(y)

def make_dataset(X, y, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X))          # no fixed seed — stochastic per fold
    ds = ds.batch(BATCH_SIZE)
    if augment:
        aug = keras.Sequential([
            keras.layers.RandomFlip('horizontal'),
            keras.layers.RandomRotation(0.1),
            keras.layers.RandomZoom(0.1),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def fresh_model(model_path):
    """
    Load architecture and optimizer config from disk, then reset ALL weights
    to random initialisation. Each fold trains from scratch — the saved file
    is used only for architecture and optimizer config, not the trained weights.
    Covers Conv2D, Dense (kernel + bias) and BatchNormalization (gamma, beta,
    moving_mean, moving_variance) so no converged state carries over.
    """
    model = tf.keras.models.load_model(model_path)
    for layer in model.layers:
        # Trainable weights: kernel and bias (Conv2D, Dense)
        if hasattr(layer, 'kernel_initializer') and layer.kernel is not None:
            layer.kernel.assign(
                layer.kernel_initializer(layer.kernel.shape, dtype=layer.kernel.dtype))
        if hasattr(layer, 'bias_initializer') and layer.bias is not None:
            layer.bias.assign(
                layer.bias_initializer(layer.bias.shape, dtype=layer.bias.dtype))
        # BatchNormalization: reset gamma, beta, moving_mean, moving_variance
        if isinstance(layer, keras.layers.BatchNormalization):
            if layer.gamma is not None:
                layer.gamma.assign(tf.ones_like(layer.gamma))
            if layer.beta is not None:
                layer.beta.assign(tf.zeros_like(layer.beta))
            layer.moving_mean.assign(tf.zeros_like(layer.moving_mean))
            layer.moving_variance.assign(tf.ones_like(layer.moving_variance))
    opt_cfg = model.optimizer.get_config()
    fresh_opt = model.optimizer.__class__.from_config(opt_cfg)
    model.compile(optimizer=fresh_opt,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ── Load fixed val set once (raw arrays; datasets rebuilt per fold) ───────────
print("Loading val images H(1)–H(50) and D(1)–D(50)...")
val_X, val_y = load_val_images(VAL_DIR, n=50)
print(f"Val: {val_X.shape[0]} images | "
      f"Healthy={np.sum(val_y==0)}, Diseased={np.sum(val_y==1)}")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_results = []

for model_path, model_label, cache_key in MODELS:
    print(f"\n{'='*60}")
    print(f"MODEL: {model_label}")
    print(f"FILE : {model_path}")
    print(f"{'='*60}")

    print("Loading training cache...")
    train_X = load_npz(os.path.join(CACHE_DIR, f'train_{cache_key}.npz'))
    train_y = load_npz(os.path.join(CACHE_DIR, f'train_labels_{cache_key}.npz'))
    print(f"Train: {train_X.shape[0]} images | "
          f"Healthy={np.sum(train_y==0)}, Diseased={np.sum(train_y==1)}")
    print(f"Value range: {train_X.min():.3f} – {train_X.max():.3f}")

    fold_val_accs = []

    for fold, (train_idx, heldout_idx) in enumerate(skf.split(train_X, train_y), start=1):
        print(f"\n  ── Fold {fold}/5 ─────────────────────────────────────")

        X_train_fold = train_X[train_idx]
        y_train_fold = train_y[train_idx]
        X_heldout    = train_X[heldout_idx]
        y_heldout    = train_y[heldout_idx]

        h_tr = np.sum(y_train_fold == 0); d_tr = np.sum(y_train_fold == 1)
        h_ho = np.sum(y_heldout == 0);    d_ho = np.sum(y_heldout == 1)
        print(f"     Train split : {len(X_train_fold)} images (H={h_tr}, D={d_tr})")
        print(f"     Held-out    : {len(X_heldout)} images (H={h_ho}, D={d_ho})")

        # Architecture from saved file, weights reset to random init
        model = fresh_model(model_path)

        # All datasets rebuilt inside the fold — no stale graph references
        train_ds   = make_dataset(X_train_fold, y_train_fold, augment=True, shuffle=True)
        heldout_ds = make_dataset(X_heldout, y_heldout, augment=False, shuffle=False)
        val_ds     = make_dataset(val_X, val_y, augment=False, shuffle=False)

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=20, min_delta=0.005,
                restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=8,
                min_lr=1e-6, verbose=0),
        ]

        # EarlyStopping monitors heldout_ds — val_ds is never seen during training
        model.fit(train_ds, epochs=EPOCHS, validation_data=heldout_ds,
                  callbacks=callbacks, verbose=1)

        # Evaluate on fixed external val set (unseen throughout training)
        _, val_acc = model.evaluate(val_ds, verbose=0)

        fold_val_accs.append(val_acc)
        print(f"     Fold {fold} → val acc: {val_acc:.4f}")

        del model, train_ds, heldout_ds, val_ds
        keras.backend.clear_session()

    avg_val = np.mean(fold_val_accs)
    std_val = np.std(fold_val_accs)

    print(f"\n  {'─'*56}")
    print(f"  {model_label}")
    for i, va in enumerate(fold_val_accs, 1):
        print(f"    Fold {i}: val acc={va:.4f}")
    print(f"  {'─'*56}")
    print(f"  Training accuracy (avg val over 5 folds): {avg_val:.4f} ± {std_val:.4f}")

    all_results.append((model_label, avg_val, std_val))

In [ ]:
print(f"\n{'='*60}")
print("FINAL SUMMARY — ALL MODELS")
print(f"{'='*60}")
print(f"{'Model':<40} {'Training Accuracy':>20}")
print(f"{'─'*60}")
for label, avg_v, std_v in all_results:
    print(f"  {label:<38} {avg_v:.4f} ± {std_v:.4f}")
print(f"{'='*60}")